In [ ]:

import subprocess
import sys

print("=" * 60)
print("Checking and installing required packages...")
print("=" * 60)

required_packages = [
    'transformers>=4.36.0',
    'torch',
    'accelerate',
    'pandas',
    'scikit-learn',
    'huggingface_hub',
    'tqdm'
]

for package in required_packages:
    package_name = package.split('>=')[0] if '>=' in package else package
    try:
        __import__(package_name)
        print(f"✓ {package_name} is already installed")
    except ImportError:
        print(f"✗ Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModel
import numpy as np
import gc
import warnings
import os
from tqdm import tqdm
from huggingface_hub import login, HfApi
import random
from datetime import datetime
warnings.filterwarnings('ignore')

print("\n" + "=" * 60)
print("Hugging Face Authentication")
print("=" * 60)

token = None
token_path = os.path.expanduser("~/.huggingface/token")

if os.path.exists(token_path):
    with open(token_path, 'r') as f:
        token = f.read().strip()
    print("✓ Using existing Hugging Face token")
else:
    print("\nPlease enter your Hugging Face access token.")
    print("Get it from: https://huggingface.co/settings/tokens")
    token = input("Enter token: ").strip()
    
    os.makedirs(os.path.dirname(token_path), exist_ok=True)
    with open(token_path, 'w') as f:
        f.write(token)

# Login
try:
    login(token=token)
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"✗ Login error: {e}")
    exit(1)

print("\n" + "=" * 60)
print("Loading datasets...")
print("=" * 60)

train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")
tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()

print(f"✓ Training samples: {len(train_data)}")
print(f"✓ Testing samples: {len(test_data)}")
print(f"✓ Languages: {len(tag_vocab)} classes")

print("\n" + "=" * 60)
print("Loading CodeLlama tokenizer...")
print("=" * 60)

MODEL_NAME = "codellama/CodeLlama-7b-hf"

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=token,
        trust_remote_code=True
    )
    print("✓ Tokenizer loaded")
except Exception as e:
    print(f"✗ Tokenizer error: {e}")
    exit(1)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("✓ Set pad_token to eos_token")

tokenizer.padding_side = "left"

def tokenize_batch(texts, max_length=256):
    """Efficient batch tokenization"""
    return tokenizer(
        texts,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

print("Tokenizing training data...")
train_texts = train_data["code"].tolist()
train_tokenized = tokenize_batch(train_texts, max_length=256)

print("Tokenizing test data...")
test_texts = test_data["code"].tolist()
test_tokenized = tokenize_batch(test_texts, max_length=256)

print("\n" + "=" * 60)
print("Creating datasets...")
print("=" * 60)

class CodeLlamaDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

class SimpleCodeLlamaClassifier(nn.Module):
    def __init__(self, num_classes, device):
        super(SimpleCodeLlamaClassifier, self).__init__()
        
        print("\nLoading model (this will take a few minutes)...")
        
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            token=token,
            trust_remote_code=True
        ).to(device)
        
        print("✓ Model loaded successfully!")
        
        self.hidden_size = self.encoder.config.hidden_size
        print(f"  Hidden size: {self.hidden_size}")
        
        for param in self.encoder.parameters():
            param.requires_grad = False
        
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        ).to(device)
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.classifier.parameters())
        
        print(f"  Total parameters: {total_params:,}")
        print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():  
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        
        last_hidden_state = outputs.last_hidden_state
        
    
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        logits = self.classifier(pooled_output)
        return logits

def run_experiment(seed, device, tag_vocab, train_data, test_data, 
                   train_tokenized, test_tokenized, save_results=True):
    """Run a single experiment with given seed"""
    print(f"\n{'='*60}")
    print(f"EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*60}")
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    label_encoder = LabelEncoder()
    label_encoder.fit(tag_vocab)
    
    train_dataset = CodeLlamaDataset(
        train_tokenized["input_ids"],
        train_tokenized["attention_mask"],
        torch.tensor(label_encoder.transform(train_data["language"]), dtype=torch.long)
    )
    
    test_dataset = CodeLlamaDataset(
        test_tokenized["input_ids"],
        test_tokenized["attention_mask"],
        torch.tensor(label_encoder.transform(test_data["language"]), dtype=torch.long)
    )
    
    num_classes = len(tag_vocab)
    model = SimpleCodeLlamaClassifier(num_classes, device)
    
    optimizer = optim.AdamW(
        model.classifier.parameters(),
        lr=1e-4,
        weight_decay=0.01
    )
    
    criterion = nn.CrossEntropyLoss()
    
    batch_size = 8
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    print(f"\nTraining configuration:")
    print(f"  Seed: {seed}")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {1e-4}")
    print(f"  Training samples: {len(train_dataset)}")
    print(f"  Test samples: {len(test_dataset)}")
    
    print(f"\n{'='*40}")
    print(f"Training for seed {seed}")
    print(f"{'='*40}")
    
    epochs = 3
    model.train()
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 40)
        
        total_loss = 0
        correct = 0
        total = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for batch_idx, batch in enumerate(progress_bar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
            avg_loss = total_loss / (batch_idx + 1)
            accuracy = correct / total
            progress_bar.set_postfix({
                'loss': f'{avg_loss:.4f}',
                'acc': f'{accuracy:.4f}'
            })
        
        avg_epoch_loss = total_loss / len(train_loader)
        epoch_accuracy = correct / total
        
        print(f"Epoch {epoch+1} Summary:")
        print(f"  Loss: {avg_epoch_loss:.4f}")
        print(f"  Accuracy: {epoch_accuracy:.4f}")
    
    if save_results:
        model_save_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_7b_seed{seed}.pth"
        torch.save({
            'model_state_dict': model.state_dict(),
            'classifier_state_dict': model.classifier.state_dict(),
            'label_encoder': label_encoder,
            'tag_vocab': tag_vocab,
            'seed': seed
        }, model_save_path)
        print(f"✓ Model saved to {model_save_path}")
    
    print(f"\n{'='*40}")
    print(f"Evaluation for seed {seed}")
    print(f"{'='*40}")
    
    model.eval()
    all_preds = []
    all_labels = []
    all_confidences = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            confidences = torch.max(probs, dim=1)[0].cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_confidences.extend(confidences)
    
    accuracy = accuracy_score(all_labels, all_preds)
    
    per_lang_acc = {}
    for i, lang in enumerate(tag_vocab):
        lang_mask = np.array(all_labels) == i
        if lang_mask.any():
            lang_accuracy = accuracy_score(
                np.array(all_labels)[lang_mask],
                np.array(all_preds)[lang_mask]
            )
            per_lang_acc[lang] = lang_accuracy
    
    print(f"\n✓ Evaluation complete for seed {seed}!")
    print(f"  Test Accuracy: {accuracy:.4f}")
    
    if save_results:
        results_df = pd.DataFrame({
            'true_label': [tag_vocab[l] for l in all_labels],
            'predicted_label': [tag_vocab[p] for p in all_preds],
            'confidence': all_confidences,
            'is_correct': [1 if p == l else 0 for p, l in zip(all_preds, all_labels)]
        })
        
        results_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_7b_results_seed{seed}.csv"
        results_df.to_csv(results_path, index=False)
        print(f"✓ Results saved to {results_path}")
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        'seed': seed,
        'accuracy': accuracy,
        'per_language_accuracy': per_lang_acc,
        'predictions': all_preds,
        'labels': all_labels,
        'confidences': all_confidences
    }

def main():
    print("\n" + "=" * 60)
    print("MULTI-SEED CODELLAMA-7B EXPERIMENT")
    print("=" * 60)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    seeds = [42, 123, 456, 789, 999]
    print(f"\nRunning experiments for {len(seeds)} seeds: {seeds}")
    
    all_results = []
    
    for i, seed in enumerate(seeds):
        print(f"\n{'#'*60}")
        print(f"Experiment {i+1}/{len(seeds)}")
        print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'#'*60}")
        
        result = run_experiment(
            seed=seed,
            device=device,
            tag_vocab=tag_vocab,
            train_data=train_data,
            test_data=test_data,
            train_tokenized=train_tokenized,
            test_tokenized=test_tokenized,
            save_results=True
        )
        
        all_results.append(result)
        
        print(f"\n✓ Experiment {i+1} completed!")
        print(f"  Seed: {seed}")
        print(f"  Accuracy: {result['accuracy']:.4f}")
    
    print("\n" + "=" * 60)
    print("SUMMARY OF ALL EXPERIMENTS")
    print("=" * 60)
    
    accuracies = [r['accuracy'] for r in all_results]
    seeds_list = [r['seed'] for r in all_results]
    
    summary_data = []
    for i, result in enumerate(all_results):
        summary_data.append({
            'Seed': result['seed'],
            'Accuracy': result['accuracy'],
            'Accuracy_%': f"{result['accuracy']*100:.2f}%"
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    mean_accuracy = np.mean(accuracies)
    std_accuracy = np.std(accuracies)
    min_accuracy = np.min(accuracies)
    max_accuracy = np.max(accuracies)
    
    print(f"\nOverall Statistics:")
    print(f"  Number of experiments: {len(all_results)}")
    print(f"  Mean Accuracy: {mean_accuracy:.4f} ({mean_accuracy*100:.2f}%)")
    print(f"  Std Accuracy: {std_accuracy:.4f}")
    print(f"  Min Accuracy: {min_accuracy:.4f} ({min_accuracy*100:.2f}%)")
    print(f"  Max Accuracy: {max_accuracy:.4f} ({max_accuracy*100:.2f}%)")
    print(f"  Range: {max_accuracy - min_accuracy:.4f}")
    
    print(f"\nDetailed Results:")
    print(summary_df.to_string(index=False))
    
    print(f"\n{'='*40}")
    print("PER-LANGUAGE ANALYSIS (Across all seeds)")
    print(f"{'='*40}")
    
    lang_accuracies = {lang: [] for lang in tag_vocab}
    
    for result in all_results:
        for lang, acc in result['per_language_accuracy'].items():
            lang_accuracies[lang].append(acc)
    
    lang_summary = []
    for lang in tag_vocab:
        if lang_accuracies[lang]:
            mean_acc = np.mean(lang_accuracies[lang])
            std_acc = np.std(lang_accuracies[lang])
            lang_summary.append({
                'Language': lang,
                'Mean_Accuracy': f"{mean_acc:.4f}",
                'Std_Accuracy': f"{std_acc:.4f}",
                'Min': f"{np.min(lang_accuracies[lang]):.4f}",
                'Max': f"{np.max(lang_accuracies[lang]):.4f}"
            })
    
    lang_df = pd.DataFrame(lang_summary)
    print("\nLanguage-wise performance across seeds:")
    print(lang_df.to_string(index=False))
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    summary_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_7b_multiseed_summary_{timestamp}.csv"
    
    detailed_summary = []
    for result in all_results:
        row = {
            'seed': result['seed'],
            'overall_accuracy': result['accuracy']
        }
        
        for lang in tag_vocab:
            row[f'acc_{lang}'] = result['per_language_accuracy'].get(lang, np.nan)
        
        detailed_summary.append(row)
    
    detailed_df = pd.DataFrame(detailed_summary)
    detailed_df.to_csv(summary_path, index=False)
    
    print(f"\n✓ Detailed summary saved to: {summary_path}")
    
    stats_summary = {
        'timestamp': timestamp,
        'num_experiments': len(all_results),
        'seeds': seeds,
        'mean_accuracy': mean_accuracy,
        'std_accuracy': std_accuracy,
        'min_accuracy': min_accuracy,
        'max_accuracy': max_accuracy,
        'accuracy_range': max_accuracy - min_accuracy
    }
    
    stats_df = pd.DataFrame([stats_summary])
    stats_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_7b_stats_summary_{timestamp}.csv"
    stats_df.to_csv(stats_path, index=False)
    
    print(f"✓ Statistics summary saved to: {stats_path}")
    
    print("\n" + "=" * 60)
    print("EXPERIMENT COMPLETE!")
    print("=" * 60)
    print(f"Total experiments run: {len(all_results)}")
    print(f"Overall mean accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
    print(f"Best seed: {seeds_list[np.argmax(accuracies)]} ({max_accuracy:.4f})")
    print(f"Worst seed: {seeds_list[np.argmin(accuracies)]} ({min_accuracy:.4f})")
    print("=" * 60)

if __name__ == "__main__":
    main()